### Step 0: Colab Setup
this part should be commented out if running locally

In [1]:
# !git clone https://github.com/IAmGreyBunny/SC3000-Lab1 #this is a fork of the original repo
# %cd NanoGPT-Math/dpo

# !gdown 1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ -O ../sft/gpt.pt
# !ls -lh ../sft

### Step 1: Install necesscary packages

In [2]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Package imports and configuration

In [3]:
import sys
import os
sys.path.append(os.path.abspath(".."))
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

In [4]:
print("device:", device, "| vocab size:", len(stoi))
print(encode("17+19=?"), "->", decode(encode("17+19=?")))
print("newline id:", stoi['\n'], "| period id:", stoi['.'], "| '!' in vocab:", '!' in stoi)

device: cuda | vocab size: 74
[13, 19, 4, 13, 21, 9, 10] -> 17+19=?
newline id: 0 | period id: 7 | '!' in vocab: False


### Step 3: Define helper functions

In [5]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [6]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

Before

In [7]:
gpt.eval()
with torch.no_grad():
    for q in ["17+19=?", "3*17=?", "72-x=34,x=?"]:
        x = torch.tensor(encode(q), dtype=torch.long, device=device)[None, ...]
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print(decode(y[0].tolist()))
gpt.train()   # back to training mode before Step 5

17+19=? Sory, I don't know.
3*17=? Sory, I don't know.
72-x=34,x=? Sorry, I don't know.


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

In [8]:
import json, random
random.seed(42)
NEG = "Sorry, I do not know"          # no '!' since it isn't in the vocab

def arith():
    op = random.choice("+-*/")
    if op == "+": a, b = random.randint(0, 99), random.randint(0, 99); ans = a + b
    elif op == "-": a = random.randint(0, 99); b = random.randint(0, a); ans = a - b
    elif op == "*": a, b = random.randint(0, 20), random.randint(0, 20); ans = a * b
    else: b = random.randint(1, 20); ans = random.randint(0, 20); a = b * ans   # exact division
    q = f"{a}{op}{b}=?"
    return q, f"{q} The answer is {ans} because {a}{op}{b} equals {ans}."

def algebra():
    form = random.choice(["x+a", "a+x", "x-a", "a-x", "x*a", "a*x", "x/a", "a/x"])
    if form in ("x+a", "a+x"):
        x, a = random.randint(0, 99), random.randint(0, 99); c = x + a; expl = f"{c}-{a}"
    elif form == "x-a":
        x = random.randint(0, 99); a = random.randint(0, x); c = x - a; expl = f"{c}+{a}"
    elif form == "a-x":
        a = random.randint(0, 99); x = random.randint(0, a); c = a - x; expl = f"{a}-{c}"
    elif form in ("x*a", "a*x"):
        x, a = random.randint(0, 20), random.randint(1, 20); c = x * a; expl = f"{c}/{a}"
    elif form == "x/a":
        c = random.randint(0, 20); a = random.randint(1, 20); x = c * a; expl = f"{c}*{a}"
    else:  # a/x = c
        x, c = random.randint(1, 20), random.randint(1, 20); a = x * c; expl = f"{a}/{c}"
    q = f"{form.replace('a', str(a))}={c},x=?"
    return q, f"{q} The answer is {x} because {expl} equals to {x}."

pairs = []
for _ in range(100000):
    q, pos = arith() if random.random() < 0.5 else algebra()
    pairs.append({"negative": f"{q} {NEG}", "positive": pos})

# sanity checks
assert all(c in stoi for p in pairs for s in p.values() for c in s)
assert max(len(s) + 4 for p in pairs for s in p.values()) <= max_length
json.dump(pairs, open("./pos_neg_pairs.json", "w"), indent=0)
print(len(pairs), pairs[:3])

100000 [{'negative': 'x+35=129,x=? Sorry, I do not know', 'positive': 'x+35=129,x=? The answer is 94 because 129-35 equals to 94.'}, {'negative': '94-13=? Sorry, I do not know', 'positive': '94-13=? The answer is 81 because 94-13 equals 81.'}, {'negative': '54+x=129,x=? Sorry, I do not know', 'positive': '54+x=129,x=? The answer is 75 because 129-54 equals to 75.'}]


In [9]:
# Load data from ./data/pos_neg_pairs.json
with open("./pos_neg_pairs.json") as f:
    lines = json.load(f)
print(f"Loaded {len(lines)} pairs, e.g. {lines[0]}")

Loaded 100000 pairs, e.g. {'negative': 'x+35=129,x=? Sorry, I do not know', 'positive': 'x+35=129,x=? The answer is 94 because 129-35 equals to 94.'}


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [10]:
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)
steps_per_epoch = len(lines) // batch_size
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs * steps_per_epoch, eta_min=base_lr * 0.1)

### Step 7: Begin training (**students are required to complete this part!**)

In [11]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)
        # preference term pushes the good answer above the refusal;
        # the extra -0.1*pos term also raises the good answer's likelihood directly
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        pbar.set_description(f"epoch {epoch} loss {loss.item():.4f} "
                             f"pos {pos_logprob.mean().item():.3f} neg {neg_logprob.mean().item():.3f}")
        ###########################################################
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

epoch 0 loss 0.0234 pos -0.234 neg -7.499: : 1562it [02:21, 11.01it/s]


Saved checkpoint to ./dpo.pt


epoch 1 loss 0.0213 pos -0.213 neg -7.732: : 1562it [02:21, 11.00it/s]


Saved checkpoint to ./dpo.pt


epoch 2 loss 0.0204 pos -0.204 neg -8.025: : 1562it [02:24, 10.80it/s]


Saved checkpoint to ./dpo.pt


epoch 3 loss 0.0188 pos -0.188 neg -8.088: : 1562it [02:26, 10.65it/s]


Saved checkpoint to ./dpo.pt


epoch 4 loss 0.0179 pos -0.178 neg -8.097: : 1562it [02:22, 10.93it/s]

Saved checkpoint to ./dpo.pt


### Step 8: Begin testing (**students are required to complete this part!**)

After

In [12]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print(decode(y[0].tolist()))
        ###########################################################

17+19=? The answer is 36 because 17+19 equals 36.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 18 because 72/4 equals 18.
72-x=34,x=? The answer is 48 because 72-34 equals to 48.
x*11=44,x=? The answer is 4 because 44/11 equals to 4.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 19 because 72/4 equals 19.
72-x=34,x=? The answer is 48 because 72-34 equals to 48.


In [13]:
import re
from collections import defaultdict

def ask(q):
    x = torch.tensor(encode(q), dtype=torch.long, device=device)[None, ...]
    y, _ = gpt.generate(x, max_new_tokens, temperature=1.0, top_k=1)
    return decode(y[0].tolist())

def answer_of(s):
    m = re.search(r"The answer is (-?\d+)", s)
    return m.group(1) if m else None

train_qs = {p["positive"].split(" ")[0] for p in lines}   # the question is everything before the first space
random.seed(123)
correct, total = defaultdict(int), defaultdict(int)
with torch.no_grad():
    for _ in range(3000):
        q, pos = arith() if random.random() < 0.5 else algebra()
        if q in train_qs:
            continue                      # only test on problems it never saw
        kind = ("equation " if "x" in q else "") + next(c for c in q if c in "+-*/")
        total[kind] += 1
        correct[kind] += answer_of(ask(q)) == answer_of(pos)

for k in sorted(total):
    print(f"{k:12s} {correct[k]}/{total[k]} = {correct[k]/total[k]:.1%}")
print(f"overall: {sum(correct.values())/sum(total.values()):.1%}")

+            102/124 = 82.3%
-            29/40 = 72.5%
equation +   134/195 = 68.7%
equation -   92/109 = 84.4%
overall: 76.3%
